# 🧠 MiniGPT-from-scratch — Colab Notebook

Train a ~30M-parameter GPT from scratch on TinyStories, on Colab's free T4 GPU, in ~1 hour.

**Repo**: https://github.com/SherlockDavis/MiniGPT-from-scratch

This notebook walks through:
1. Verify GPU
2. Clone the repo and install dependencies
3. Download the TinyStories dataset
4. Smoke test (100 training steps, ~1 min)
5. (Optional) Full training (10000 steps, ~83 min)
6. Generate text from a trained checkpoint

> ⚠️ **Before running**: go to **Runtime → Change runtime type → Hardware accelerator → T4 GPU**, otherwise training will be unusably slow on CPU.


## 1. Verify GPU

`nvidia-smi` should print info for a Tesla T4 (16 GB). If it errors, you're not on a GPU runtime.


In [ ]:
!nvidia-smi

## 2. Clone repo + install dependencies

The clone is shallow (`--depth=1`) — we only need the latest commit, not the full history.


In [ ]:
!git clone --depth=1 https://github.com/SherlockDavis/MiniGPT-from-scratch.git
%cd MiniGPT-from-scratch


In [ ]:
!pip install -q -r requirements.txt

## 3. Download TinyStories

The script downloads `TinyStoriesV2-GPT4-train.txt` (~2 GB) and `TinyStoriesV2-GPT4-valid.txt` (~20 MB) into `data/`. Takes 2–5 min on Colab's network.


In [ ]:
!bash scripts/download_data.sh

## 4. Smoke test (100 steps, ~1 min)

The first run also tokenizes the corpus into `data/train.bin` / `data/val.bin` — that one-time prep takes ~3 min on a fast tokenizer. Subsequent runs reuse the cached `.bin` files.

Watch for:
- `loss` should drop from ~11 (random init) toward ~7 within 100 steps
- No `Non-finite loss` errors
- Step time around 0.5 s on T4


In [ ]:
!python train.py --max-iters 100 --no-wandb

## 5. Full training (~83 min on T4)

This runs the canonical 10000-step training with FP16 AMP + cosine LR schedule. Final val PPL should be around 4.7.

> 💡 If you don't want to wait, **skip this cell** and use the pre-trained checkpoint by downloading it from the repo's GitHub Release (or just rely on the smoke test for a much weaker model).
>
> 💡 To use Weights & Biases, drop `--no-wandb` and run `wandb login` first (you'll need a free account at https://wandb.ai).


In [ ]:
!python train.py --no-wandb

## 6. Generate text

After training, the latest checkpoint is at `checkpoints/ckpt_step10000.pt`. Try the four sampling modes:


In [ ]:
# Greedy — deterministic, "safest" output
!python generate.py --checkpoint checkpoints/ckpt_step10000.pt \
    --prompt "Once upon a time" --max-new-tokens 200 \
    --temperature 0.0 --seed 42


In [ ]:
# Temperature 0.8 — recommended for narrative
!python generate.py --checkpoint checkpoints/ckpt_step10000.pt \
    --prompt "Once upon a time" --max-new-tokens 200 \
    --temperature 0.8 --seed 42


In [ ]:
# Top-p 0.9 nucleus sampling — adaptive cutoff
!python generate.py --checkpoint checkpoints/ckpt_step10000.pt \
    --prompt "Once upon a time" --max-new-tokens 200 \
    --temperature 1.0 --top-p 0.9 --seed 42


In [ ]:
# Try your own prompt
!python generate.py --checkpoint checkpoints/ckpt_step10000.pt \
    --prompt "The little dragon" --max-new-tokens 200 \
    --temperature 0.8 --top-p 0.9 --seed 7


## 7. (Optional) Save the checkpoint to Google Drive

Colab session storage is wiped when the runtime disconnects. To keep your trained checkpoint, mount Drive and copy it over:


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/MiniGPT
!cp checkpoints/ckpt_step10000.pt /content/drive/MyDrive/MiniGPT/
!ls -lh /content/drive/MyDrive/MiniGPT/


## What's next?

- Read the technical blog series in [`blog/`](https://github.com/SherlockDavis/MiniGPT-from-scratch/tree/main/blog) — three parts covering architecture, training, and generation.
- Try editing `config.py` to scale the model up (`n_layer=12, n_embd=768` → 124M, GPT-2 small).
- Add a `repetition_penalty` to `generate.py` to fix the model's mild repeat tendency.

If this was useful, **⭐ the repo on [GitHub](https://github.com/SherlockDavis/MiniGPT-from-scratch)**!
